# 🌍 English ↔ Malayalam Translation API (FastAPI + Transformers)

This notebook sets up a **FastAPI server** that can **translate between English and Malayalam** using the **Helsinki-NLP MarianMT models** from Hugging Face.

## 🚀 Features

- Translate between:
  - English → Malayalam (`en-ml`)
  - Malayalam → English (`ml-en`)
- Uses **cached models**, loaded once at startup for fast, efficient response
- Based on `transformers` (`MarianMTModel`, `MarianTokenizer`)
- Ready for deployment via **Google Colab** + **ngrok**
- Simple API with `POST /translate`

---

## 🛠️ API Usage

### ▶️ Endpoint
```bash
POST /translate


In [ ]:
!pip install fastapi uvicorn transformers torch pyngrok nest-asyncio pyngrok sacremoses -q

In [2]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import MarianMTModel, MarianTokenizer
from typing import Literal
import torch
from pyngrok import ngrok
import nest_asyncio
import uvicorn
from google.colab import userdata


In [3]:
import subprocess

NGROK_AUTH_TOKEN = userdata.get('NGROK_TOKEN')
subprocess.run(["ngrok", "config", "add-authtoken", NGROK_AUTH_TOKEN])


CompletedProcess(args=['ngrok', 'config', 'add-authtoken', '2vXANeCXuNspT7HJZCzpaURK7Fi_23x96rkMpg2moPAYhKNne'], returncode=0)

In [8]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import MarianMTModel, MarianTokenizer
from typing import Literal
import torch
from contextlib import asynccontextmanager

model_cache = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Preload models here
    model_pairs = [("en", "ml"), ("ml", "en")]
    for src, tgt in model_pairs:
        key = f"{src}-{tgt}"
        try:
            model_name = f"Helsinki-NLP/opus-mt-{src}-{tgt}"
            tokenizer = MarianTokenizer.from_pretrained(model_name)
            model = MarianMTModel.from_pretrained(model_name)
            model_cache[key] = (tokenizer, model)
            print(f"✅ Loaded model: {key}")
        except Exception as e:
            print(f"❌ Failed to load model {key}: {e}")
    yield  # app is running
    # You could clean up resources here after shutdown

app = FastAPI(title="Translation API", lifespan=lifespan)

class TranslationRequest(BaseModel):
    text: str
    direction: Literal["en-ml", "ml-en"]

def get_model(source_lang: str, target_lang: str):
    key = f"{source_lang}-{target_lang}"
    if key not in model_cache:
        raise HTTPException(status_code=500, detail=f"Model {key} not loaded.")
    return model_cache[key]

def translate(text: str, source_lang: str, target_lang: str) -> str:
    tokenizer, model = get_model(source_lang, target_lang)
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        translated_tokens = model.generate(**inputs)
    return tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

@app.post("/translate")
def translate_route(request: TranslationRequest):
    lang_map = {
        "en-ml": ("en", "ml"),
        "ml-en": ("ml", "en")
    }
    if request.direction not in lang_map:
        raise HTTPException(status_code=400, detail="Invalid direction")

    src, tgt = lang_map[request.direction]
    translated = translate(request.text, src, tgt)
    return {"translated_text": translated}

In [ ]:
nest_asyncio.apply()

# Open a tunnel to the API using ngrok
public_url = ngrok.connect(8000)
print("Your public FastAPI URL:", public_url)

# Run the server
uvicorn.run(app, host="0.0.0.0", port=8000)


## 🔤 Request Body (JSON)

```
{
  "text": "Hello, how are you?",
  "direction": "en-ml"  // or "ml-en"
}

```



```
{
  "translated_text": "ഹലോ, സുഖമാണോ?"
}

```





## Example: External Python Client

In [ ]:
import requests

url = "http://{ngrok_url}/translate"
payload = {"text": "ഹലോ!", "direction": "ml-en"}

res = requests.post(url, json=payload)
print(res.json())  # {'translated_text': 'Hello!'}